# 06 · Export: gold tables as CSV

Writes every gold table to a single CSV file, so the data outlives the Fabric trial capacity.

- **Input:** `lh_gold.dbo` (`fact_ad`, `fact_ad_skill`, `dim_date`, `dim_role`, `dim_location`, `dim_employer`, `dim_skill`)
- **Output:** `Files/export/<table>.csv` in `lh_gold`, UTF-8, comma-separated, with header
- **How to run:** *Run all*, then download the files from the lakehouse.
- **Lakehouse:** `lh_gold` attached as default.

The CSV files are the source of the `.pbix` version of the report and are published in the repo's `data/` folder. The tables are small (largest ~46,000 rows), so each one is converted to pandas and written as one file rather than as Spark's folder of part files.

In [ ]:
from pathlib import Path

TABLES = ["fact_ad", "fact_ad_skill", "dim_date", "dim_role",
          "dim_location", "dim_employer", "dim_skill"]
EXPORT_DIR = Path("/lakehouse/default/Files/export")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

for name in TABLES:
    df = spark.table(f"lh_gold.dbo.{name}").toPandas()
    out = EXPORT_DIR / f"{name}.csv"
    df.to_csv(out, index=False, encoding="utf-8")
    print(f"{name:15} {len(df):>7} rows  {out.stat().st_size / 1024:>8.0f} KB")

In [ ]:
# Verify: every file exists and has as many rows as its table
import pandas as pd

for name in TABLES:
    n_csv = len(pd.read_csv(EXPORT_DIR / f"{name}.csv"))
    n_tbl = spark.table(f"lh_gold.dbo.{name}").count()
    assert n_csv == n_tbl, f"{name}: {n_csv} rows in CSV, {n_tbl} in table"
print("All exports match their tables.")